# Two-Way MANOVA Nedir?
Two-Way MANOVA, One-Way MANOVA'nın (bir önceki konumuz) **iki faktörlü** hali — tıpkı Two-Way ANOVA'nın One-Way ANOVA'ya eklediği gibi (Konu 68), burada da **iki bağımsız değişken** (örn: Kanal VE Segment), **birden fazla bağımlı değişkeni** (örn: Memnuniyet VE Sadakat) aynı anda etkiler mi diye test ediyoruz.

## Test Edilen 3 Hipotez (Two-Way ANOVA'daki Gibi, Ama Çok Değişkenli)
1. **Ana Etki 1:** Kanal, bağımlı değişkenlerin kombinasyonunu etkiliyor mu?
2. **Ana Etki 2:** Segment, bağımlı değişkenlerin kombinasyonunu etkiliyor mu?
3. **Etkileşim:** Kanal ve Segment'in birlikte etkisi, bağımlı değişkenlerin kombinasyonunda fark yaratıyor mu?

## Python'da Kullanımı
```python
from statsmodels.multivariate.manova import MANOVA

model = MANOVA.from_formula('Memnuniyet + Sadakat ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', data=df)
print(model.mv_test())
```
Formül yapısı, Two-Way ANOVA'daki (Konu 68) formülle birebir aynı mantık 
— sadece sol tarafta artık tek değil, iki bağımlı değişken var.

## Sonrasında Ne Yapılır?
Herhangi bir terim (Kanal, Segment, veya etkileşim) anlamlı çıkarsa, One-Way MANOVA'da yaptığımız gibi, **her bağımlı değişken için ayrı ayrı Two-Way ANOVA** çalıştırarak, farkın hangi değişkenden ve hangi terimden (ana etki mi, etkileşim mi) geldiğini araştırırız.

In [2]:
import numpy as np
import pandas as pd
from statsmodels.multivariate.manova import MANOVA
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(42)
n = 20

# --- Veri Oluşturma: Kanal x Segment = 3x2 = 6 grup, her birinden 20 gözlem ---
kombinasyonlar = {
    ('Telefon', 'Bireysel'):      {'memnuniyet': 60, 'sadakat': 45},
    ('Telefon', 'Kurumsal'):      {'memnuniyet': 68, 'sadakat': 55},
    ('Canlı Sohbet', 'Bireysel'): {'memnuniyet': 88, 'sadakat': 60},
    ('Canlı Sohbet', 'Kurumsal'): {'memnuniyet': 78, 'sadakat': 50},
    ('Email', 'Bireysel'):        {'memnuniyet': 65, 'sadakat': 48},
    ('Email', 'Kurumsal'):        {'memnuniyet': 72, 'sadakat': 52},
}

kanal_l, segment_l, memnuniyet_l, sadakat_l = [], [], [], []
for (kanal, segment), ort in kombinasyonlar.items():
    kanal_l += [kanal]*n
    segment_l += [segment]*n
    memnuniyet_l += list(np.random.normal(ort['memnuniyet'], 7, n))
    sadakat_l += list(np.random.normal(ort['sadakat'], 7, n))

df = pd.DataFrame({'Kanal': kanal_l, 'Segment': segment_l, 
                    'Memnuniyet': memnuniyet_l, 'Sadakat': sadakat_l})

# H0-1: Kanal, Memnuniyet+Sadakat kombinasyonunu etkilemez.
# H0-2: Segment, Memnuniyet+Sadakat kombinasyonunu etkilemez.
# H0-3: Kanal ve Segment arasında etkileşim yoktur.

# --- Adım 1: Two-Way MANOVA ---
manova_model = MANOVA.from_formula(
    'Memnuniyet + Sadakat ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', 
    data=df
)
print(manova_model.mv_test())

# --- Adım 2: Takip Testi - MEMNUNIYET için ayrı Two-Way ANOVA ---
model_mem = ols('Memnuniyet ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', data=df).fit()
print(sm.stats.anova_lm(model_mem, typ=2))

# --- Adım 3: Takip Testi - SADAKAT için ayrı Two-Way ANOVA ---
model_sad = ols('Sadakat ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', data=df).fit()
print(sm.stats.anova_lm(model_sad, typ=2))

                   Multivariate linear model
                                                                
----------------------------------------------------------------
       Intercept         Value  Num DF  Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda  0.0201 2.0000 113.0000 2747.8394 0.0000
         Pillai's trace  0.9799 2.0000 113.0000 2747.8394 0.0000
 Hotelling-Lawley trace 48.6343 2.0000 113.0000 2747.8394 0.0000
    Roy's greatest root 48.6343 2.0000 113.0000 2747.8394 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
          C(Kanal)        Value  Num DF  Den DF  F Value  Pr > F
----------------------------------------------------------------
            Wilks' lambda 0.2758 4.0000 226.0000  51.0803 0.0000
           Pillai's trace 0.7273 4.0000 228.0

### Sonuç
Kanal ve Segment'in, Memnuniyet ve Sadakat kombinasyonu üzerindeki etkisini test etmek için Two-Way MANOVA uyguladık. Sonuçlara göre Kanal'ın ana etkisi (Wilks' Λ=0.276, p<0.001), Segment'in ana etkisi (Wilks' Λ=0.689, p<0.001) ve Kanal-Segment etkileşimi (Wilks' Λ=0.491, p<0.001) istatistiksel olarak anlamlı bulunmuştur.

MANOVA'nın hangi bağımlı değişkenden kaynaklandığını netleştirmek için, Memnuniyet ve Sadakat için ayrı ayrı Two-Way ANOVA uyguladık. Sonuçlara göre Kanal'ın ve Kanal-Segment etkileşiminin, hem Memnuniyet (sırasıyla p<0.001 ve p<0.001) hem Sadakat (sırasıyla p<0.001 ve p<0.001) üzerinde ayrı ayrı da anlamlı etkileri olduğu görülmüştür.

Dikkat çekici bir bulgu olarak, Segment'in ana etkisi MANOVA'da anlamlı çıkmasına rağmen, ne Memnuniyet'te (p=0.103) ne de Sadakat'te (p=0.078) tek başına anlamlı bulunamamıştır. Bu durum, Segment'in her iki değişkende ayrı ayrı zayıf kalan etkisinin, MANOVA'nın çok değişkenli yapısında birleşerek anlamlı hale geldiğini göstermektedir — bu da MANOVA'nın, tek tek ANOVA'ların kaçırabileceği ince/birleşik etkileri yakalayabildiğinin somut bir kanıtıdır.

İş açısından, destek kanalı seçimi hem müşteri memnuniyetini hem sadakatini güçlü şekilde etkilemekte ve bu etki müşteri segmentine göre 
de değişmektedir. Bu nedenle kanal stratejisi, tüm segmentlere aynı şekilde değil, segment bazlı optimize edilmelidir.